<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

Working dir: /content/Applied-Search-Intelligence-System/Applied-Search-Intelligence-System


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Why retrain on the full population here:** w05/w06 already validated this model honestly (a
client-grouped held-out split, a leakage audit, a before/after comparison). This playbook is the
*deployed* queue, not another validation run, so it's fair, standard practice to retrain the same
Logistic Regression pipeline on all 30,000 rows before generating recommendations. The precision@K
numbers this playbook's confidence rests on are still the held-out ones from w05/w06, not
recomputed here.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [17]:
RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW_PATH)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Same honest feature set validated in w05/w06.
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc", "word_count", "char_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier", "position_tier",
    "provider_used", "model_used",
]
X = df[numeric_features + categorical_features]
y = df["is_declining_label"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
pipe = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=2000, random_state=42))])
pipe.fit(X, y)
df["decline_probability"] = pipe.predict_proba(X)[:, 1]

print(f"Rows: {len(df):,}")
print(df["decline_probability"].describe().round(3))

Rows: 30,000
count    30000.000
mean         0.542
std          0.180
min          0.009
25%          0.427
50%          0.564
75%          0.689
max          1.000
Name: decline_probability, dtype: float64


**Reason codes: restricted to actionable features on purpose.** I compute each row's top
contributing feature (coefficient x standardized value) toward the "declining" class, but I
deliberately restrict the *candidate* features to ones an editor can actually act on: staleness,
CTR, position, engagement, scroll depth, word count, and visibility/traffic consistency. I
excluded `content_type` and `model_used` from the reason-code pool on purpose; more on why in
section 3's no-go list.

In [18]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label',
       'decline_probability'],
      dtype='object')

In [19]:
actionable_numeric = [
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "word_count", "days_with_impressions", "days_with_sessions", "sessions_90d",
    "content_age_days",
]
reason_labels = {
    "days_since_last_update": "stale_content",
    "ctr": "weak_click_through",
    "avg_position": "weak_ranking_position",
    "engagement_rate": "weak_engagement",
    "scroll_rate": "weak_scroll_depth",
    "word_count": "content_depth_mismatch",
    "days_with_impressions": "past_visibility_peak",
    "days_with_sessions": "inconsistent_traffic",
    "sessions_90d": "large_page_regression_risk",
    "content_age_days": "aging_past_lifecycle_peak",
}

num_prep = pipe.named_steps["prep"].named_transformers_["num"]
Xnum_t = num_prep.transform(X[numeric_features])
clf = pipe.named_steps["clf"]
num_coefs = clf.coef_[0][:len(numeric_features)]

actionable_idx = [numeric_features.index(f) for f in actionable_numeric]
contributions = Xnum_t[:, actionable_idx] * num_coefs[actionable_idx]
top_local_idx = np.argmax(contributions, axis=1)
df["reason_code"] = [reason_labels[actionable_numeric[i]] for i in top_local_idx]

df["reason_code"].value_counts()

,count
reason_code,
past_visibility_peak,13725
inconsistent_traffic,7793
aging_past_lifecycle_peak,3127
content_depth_mismatch,2494
large_page_regression_risk,2032
weak_ranking_position,401
weak_scroll_depth,240
stale_content,171
weak_engagement,9


Two of these deserve a plain-words note, because the direction is *not* what intuition
suggests, and the honest explanation ties back to the FlyRank paper I read in w06:

- **`past_visibility_peak`** (the single largest reason group, 13,725 rows) fires when a page has
  been visible for *most* of the 90-day window, more visible days, not fewer, pushes toward
  the decline prediction here. That matches the paper's own content-lifecycle finding (Health
  Score peaks at 61-90 days, then declines): a page visible the whole window has likely already
  finished its growth phase and is aging into the plateau-or-decline part of its lifecycle. This
  is a maturity signal, not an alarm.
- **`large_page_regression_risk`** (`sessions_90d` as the top driver) fires on already-big pages.
  This plausibly reflects *regression to the mean* as much as any real risk: a page with an
  unusually large recent session count has more room to fall back toward average than a small
  page does. I'm flagging this honestly rather than implying big pages are somehow being
  penalized.

In [20]:
# Ranked action tiers, by score percentile.
q90 = df["decline_probability"].quantile(0.90)
q70 = df["decline_probability"].quantile(0.70)

def action_tier(p):
    if p >= q90:
        return "PRIORITY_REVIEW"
    if p >= q70:
        return "SCHEDULE_REVIEW"
    return "MONITOR"

df["action"] = df["decline_probability"].apply(action_tier)
print(f"Thresholds -- top 10%: {q90:.3f}   top 30%: {q70:.3f}")
df["action"].value_counts()

Thresholds -- top 10%: 0.757   top 30%: 0.666


,count
action,
MONITOR,21000
SCHEDULE_REVIEW,6000
PRIORITY_REVIEW,3000


**The decay/refresh insight, and where the archetypes come from.** My own w04 signal check
found the 91-180-day-since-update bucket declining at 61% versus a 51% base rate for freshly
updated pages,  directional evidence that staleness matters, on large samples. The FlyRank paper
independently reports the same shape at portfolio scale (health peaks at 61-90 days, decays hard
by 271-365 days, and recovers sharply when refreshed). Both point the same way, on different data, that convergence is worth more than either result alone. I use a light exploratory clustering
(not another validated model, purely descriptive, the same spirit as the paper's own "Content
Archetypes" appendix) to turn that insight into a segment an editor can act on directly.

In [21]:
from sklearn.cluster import KMeans

archetype_features = ["impressions_90d", "days_since_last_update", "ctr", "avg_position", "engagement_rate"]
Xa = df[archetype_features].copy()
Xa["impressions_90d"] = np.log1p(Xa["impressions_90d"])
Xa = Xa.fillna(Xa.median())
Xa_scaled = StandardScaler().fit_transform(Xa)

km = KMeans(n_clusters=4, random_state=42, n_init=10)
df["archetype_id"] = km.fit_predict(Xa_scaled)

profile = df.groupby("archetype_id").agg(
    n=("content_id", "size"),
    avg_impressions=("impressions_90d", "mean"),
    avg_days_since_update=("days_since_last_update", "mean"),
    avg_ctr=("ctr", "mean"),
    avg_position=("avg_position", "mean"),
    avg_engagement=("engagement_rate", "mean"),
    decline_rate=("is_declining_label", "mean"),
    avg_score=("decline_probability", "mean"),
).round(2)
profile

,n,avg_impressions,avg_days_since_update,avg_ctr,avg_position,avg_engagement,decline_rate,avg_score
archetype_id,,,,,,,,
0,20113,4289.05,18.70,0.34,15.72,1.64,0.51,0.52
1,9269,7481.62,106.07,0.25,17.78,1.81,0.61,0.61
2,486,818.09,38.65,1.35,17.31,52.50,0.54,0.50
3,132,4.20,36.68,41.35,6.23,6.14,0.14,0.15


**Naming the archetypes from these real profiles** (descriptive labels, not a validated
taxonomy  exploratory, like the paper's own cluster appendix):

| ID | Name | Profile | Recommended angle |
|---|---|---|---|
| 0 | **Steady & Visible** (n=20,113, 67%) | High impressions, recently updated (~19 days), near-base decline rate | Routine monitoring; no urgent action |
| 1 | **Visible But Stale** (n=9,269, 31%) | Highest impressions of any group, but 106 days since update on average, decline rate 0.61 (highest) | **The refresh playbook's primary target** -- large audience already earned, staleness the clearest lever |
| 2 | **High-Engagement Niche** (n=486, 1.6%) | Lower volume, but CTR 1.35% and engagement 52.5% -- both far above every other group | Protect and study -- worth understanding *why* these work, not a decline risk |
| 3 | **Near-Zero-Data Edge Cases** (n=132, 0.4%) | Average 4 impressions/90d, CTR reads as 41% (a 1-2-click-on-almost-nothing artifact) | **Exclude from automated scoring entirely** -- see section 3 |

Archetype 3 is the same "rates need denominators" trap I found auditing signals in w04 and again
in w05's false-negative example, it resurfaces here as its own cluster. That's a useful
confirmation the pattern is real and recurring, not a one-off.

In [22]:
# Cost/value thinking: raw probability rank vs. economic-value-weighted rank.
# Following the paper's own "clicks x CPC, not impressions x CPC" rule (Finding #9).
df["click_equivalent_value"] = df["clicks_90d"] * df["cpc"].fillna(df["cpc"].median())
df["value_weighted_score"] = df["decline_probability"] * df["click_equivalent_value"]

top_by_probability = df.sort_values("decline_probability", ascending=False).head(5)
top_by_value = df.sort_values("value_weighted_score", ascending=False).head(5)

print("Top 5 by raw decline probability, their click-equivalent value:")
print(top_by_probability[["content_id", "decline_probability", "click_equivalent_value"]].to_string(index=False))
print("\nTop 5 by value-weighted score:")
print(top_by_value[["content_id", "decline_probability", "click_equivalent_value", "value_weighted_score"]].to_string(index=False))
print(f"\nOverlap between the two top-5 lists: {len(set(top_by_probability.content_id) & set(top_by_value.content_id))} pages")

Top 5 by raw decline probability, their click-equivalent value:
          content_id  decline_probability  click_equivalent_value
content_4560b0a818ab             1.000000                     0.0
content_8e7ba84a972b             0.999950                     0.0
content_c8ad1f4d0e56             0.999262                     0.0
content_a22b7f6c73c5             0.997366                     0.0
content_70b8f5323e29             0.987721                     0.0

Top 5 by value-weighted score:
          content_id  decline_probability  click_equivalent_value  value_weighted_score
content_3430a8b94511             0.406255                 3550.80           1442.528543
content_251ab03c2530             0.234714                 3456.81            811.362033
content_2db251d1a841             0.660303                 1133.54            748.480408
content_adddad39251c             0.281792                 2374.74            669.183529
content_e617a6dc2bf8             0.235838                 2542.65   

**This is the whole argument for cost/value thinking in one table.** Every one of the top 5
pages by raw decline probability has **zero clicks in 90 days**, these are pages nobody was
clicking on anyway, so "declining" barely means anything economically for them. The
value-weighted ranking surfaces a completely different, non-overlapping set of pages, ones that
actually carry real click and CPC value at risk. **For the actual playbook queue, I sort by
value-weighted score, not raw probability**, exactly because raw probability alone would spend an
editor's first hour on pages worth nothing.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content team's triage aid for deciding which pages to look at *first* during a
refresh cycle. It ranks and explains; it does not decide.

**Limits, stated plainly (claim-ladder language throughout):**

- **The label is a proxy, not ground truth.** `is_declining_label` is a threshold on a 30-day-vs-
  prior-30-day impression swing, w05's error analysis found this genuinely noisy at low
  impression counts (the false-negative example there had exactly 1 impression total).
- **Validated on one held-out split.** w06 measured this pipeline on a single client-grouped
  80/20 split. A naive random split on the same data showed precision@20 of 0.90 versus the
  grouped split's honest 0.65, a reminder that a single split's number is a real but narrow
  data point, not a guarantee that holds for every future client.
- **Cross-sectional, one snapshot.** This is one point-in-time export, not a time series. Nothing
  here supports "refreshing this page will increase its traffic", the honest, decision-support
  form is "this page looks worth reviewing first, because...".
- **Not validated on a brand-new client out of the box.** Per the w06 finding above, confidence
  should be lower for any client not represented in the training population until locally spot-
  checked.
- **This does not model Google's ranking algorithm.** It models *this portfolio's* observed
  performance patterns. No claim here is a claim about how search engines work.

In [23]:
print("Section 2 is a reasoning/policy statement -- see markdown above. No new computation needed.")

Section 2 is a reasoning/policy statement -- see markdown above. No new computation needed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a human should check:**
1. The page isn't already intentionally deprecated, redirected, or scheduled for removal.
2. The reason code actually holds up on a manual look at the page (a `stale_content` flag on a
   page that was just quietly updated outside this snapshot's window is a stale *read*, not a
   stale page).
3. Impressions/clicks aren't near the reliability floor (see Archetype 3, below), a tiny
   denominator can make any rate look extreme.
4. The page isn't seasonal, and this isn't just its expected off-season dip.

**What should NOT be automated ever, on this project's evidence:**

- **No auto-editing, auto-publishing, auto-redirecting, or auto-deleting content** from this
  score alone. Every action here routes through a human editor.
- **No acting on Archetype 3 (Near-Zero-Data Edge Cases) rows at all**, automated or manual-
  priority. Their CTR and decline numbers are built on 1-2 events over almost no impressions,
  the same denominator trap flagged in w04 and w05. These rows should be excluded from the queue
  entirely, not just deprioritized.
- **No using `content_type` or `model_used` to justify deprioritizing, penalizing, or singling
  out a specific authoring pipeline or AI model.** Both showed up among the model's contributing
  features in this dataset, but the FlyRank paper's own debunked myth #5 found no blanket
  AI-model penalty once age is controlled for, and my sample sizes per model are small and
  uneven (1,598 `gpt-5-mini` pages vs. 13,271 `gemini-3-flash-preview` pages), nowhere near
  enough to support a claim about one model's output quality. This is exactly why these two
  features were excluded from the actionable `reason_code` pool in section 1.
- **No treating "declining" as "will decline."** The claim this system can actually support is
  "this page is associated with the pattern our data calls declining" not a prediction of what
  will happen next, and not a causal claim about why.

In [24]:
# Concrete counts backing the no-go list above.
model_used_counts = df["model_used"].value_counts()
archetype3_count = (df["archetype_id"] == 3).sum()
print("model_used sample sizes (why model-specific claims are unsupported):")
print(model_used_counts)
print(f"\nArchetype 3 (excluded from all action) row count: {archetype3_count}")

model_used sample sizes (why model-specific claims are unsupported):
model_used
gemini-3-flash-preview    13271
gpt-4o-mini                4981
gemini-2.5-flash           3665
gpt-5-mini                 1598
unknown                     752
Name: count, dtype: int64

Archetype 3 (excluded from all action) row count: 132


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision drift:** re-measure precision@20/50/100 monthly on newly-resolved rows (pages where
  a subsequent snapshot confirms the trend). Alert if it falls meaningfully below the w05/w06
  held-out numbers (0.650 / 0.640 / 0.670)  that's the number this system has to keep beating.
- **Base-rate drift:** the population decline rate here is 0.542. A meaningful shift in that
  number (up or down) signals either a real portfolio change or a label-definition/upstream data
  change investigate before trusting the score either way.
- **Reason-code / archetype distribution drift:** if Archetype 3 (the near-zero-data edge case)
  starts growing sharply as a share of the population, that's more likely a data-quality problem
  upstream (broken tracking, indexing issues) than a real content trend treat it as a data
  alert, not a content alert.
- **New-client trigger:** per the w06 grouped-split finding, any client not represented in the
  training population should get a local spot-check on a small sample before the queue is trusted
  for them, don't wait for the quarterly cycle.
- **Retrain cadence:** quarterly by default, or immediately after any of the above triggers fire.

In [25]:
print(f"Current population base rate (retrain trigger reference number): {df['is_declining_label'].mean():.3f}")
print(f"Held-out precision@K to beat (from w05/w06): 0.650 / 0.640 / 0.670")

Current population base rate (retrain trigger reference number): 0.542
Held-out precision@K to beat (from w05/w06): 0.650 / 0.640 / 0.670


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on
these files.*

In [26]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Path("work/outputs").mkdir(parents=True, exist_ok=True)
Path("work/figures").mkdir(parents=True, exist_ok=True)

# --- 1. The ranked queue CSV (gitignored by design -- regenerated by this notebook) ---
export_cols = [
    "content_id", "client_id", "decline_probability", "value_weighted_score",
    "reason_code", "action", "archetype_id",
    "days_since_last_update", "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "click_equivalent_value", "is_declining_label",
]
queue = df.sort_values("value_weighted_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
queue_path = Path("work/outputs/action_playbook_queue.csv")
queue[["rank"] + export_cols].to_csv(queue_path, index=False)
print(f"Wrote {queue_path} -- {len(queue):,} rows")

Wrote work/outputs/action_playbook_queue.csv -- 30,000 rows


In [27]:
# --- 2. Metrics JSON (committed -- the receipts the paper's numbers trace back to) ---
metrics = {
    "population": {
        "rows": int(len(df)),
        "base_rate_declining": round(float(df["is_declining_label"].mean()), 3),
        "unique_clients": int(df["client_id"].nunique()),
    },
    "held_out_validation_reference": {
        "source": "w05/w06 client-grouped 80/20 split",
        "precision_at_20": 0.650,
        "precision_at_50": 0.640,
        "precision_at_100": 0.670,
        "random_split_precision_at_20_for_comparison": 0.900,
    },
    "action_tiers": df["action"].value_counts().to_dict(),
    "action_thresholds": {"top_10pct": round(float(q90), 3), "top_30pct": round(float(q70), 3)},
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "archetypes": {
        str(k): {
            "n": int(v["n"]), "avg_impressions": float(v["avg_impressions"]),
            "avg_days_since_update": float(v["avg_days_since_update"]),
            "avg_ctr": float(v["avg_ctr"]), "decline_rate": float(v["decline_rate"]),
        }
        for k, v in profile.to_dict("index").items()
    },
    "cost_value_check": {
        "top5_by_probability_click_equiv_value_sum": float(top_by_probability["click_equivalent_value"].sum()),
        "top5_by_value_weighted_click_equiv_value_sum": float(top_by_value["click_equivalent_value"].sum()),
        "overlap_between_top5_lists": len(set(top_by_probability.content_id) & set(top_by_value.content_id)),
    },
}
metrics_path = Path("work/outputs/action_playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote {metrics_path}")
print(json.dumps(metrics, indent=2)[:800])

Wrote work/outputs/action_playbook_metrics.json
{
  "population": {
    "rows": 30000,
    "base_rate_declining": 0.542,
    "unique_clients": 32
  },
  "held_out_validation_reference": {
    "source": "w05/w06 client-grouped 80/20 split",
    "precision_at_20": 0.65,
    "precision_at_50": 0.64,
    "precision_at_100": 0.67,
    "random_split_precision_at_20_for_comparison": 0.9
  },
  "action_tiers": {
    "MONITOR": 21000,
    "SCHEDULE_REVIEW": 6000,
    "PRIORITY_REVIEW": 3000
  },
  "action_thresholds": {
    "top_10pct": 0.757,
    "top_30pct": 0.666
  },
  "reason_code_counts": {
    "past_visibility_peak": 13725,
    "inconsistent_traffic": 7793,
    "aging_past_lifecycle_peak": 3127,
    "content_depth_mismatch": 2494,
    "large_page_regression_risk": 2032,
    "weak_ranking_position": 401,
    "weak_scroll_depth": 240,
    "


In [28]:
# --- 3. A figure to reuse in the paper: action tiers and archetype sizes ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

action_order = ["PRIORITY_REVIEW", "SCHEDULE_REVIEW", "MONITOR"]
action_counts = df["action"].value_counts().reindex(action_order)
axes[0].bar(action_order, action_counts.values, color=["#c0392b", "#e67e22", "#7f8c8d"])
axes[0].set_title("Action tiers (by decline_probability percentile)")
axes[0].set_ylabel("Pages")
for i, v in enumerate(action_counts.values):
    axes[0].text(i, v + 300, f"{v:,}", ha="center")

archetype_names = {
    0: "Steady &\nVisible", 1: "Visible\nBut Stale",
    2: "High-Engagement\nNiche", 3: "Near-Zero-Data\nEdge Cases",
}
sizes = profile["n"]
labels = [archetype_names[i] for i in sizes.index]
axes[1].bar(labels, sizes.values, color="#2980b9")
axes[1].set_title("Content archetypes (exploratory clustering)")
axes[1].set_ylabel("Pages")
for i, v in enumerate(sizes.values):
    axes[1].text(i, v + 300, f"{v:,}", ha="center")

plt.tight_layout()
fig_path = Path("work/figures/action_playbook_overview.png")
plt.savefig(fig_path, dpi=150)
plt.close()
print(f"Wrote {fig_path}")

Wrote work/figures/action_playbook_overview.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Explains intended use (content-team triage aid, decision-support only)
- [x] Ranked actions with reason codes (restricted to actionable, non-discriminatory features)
- [x] Archetype → action mapping, tied to the decay/refresh insight from w04 and the paper
- [x] Human review checklist and an explicit no-go list
- [x] Monitoring / retrain triggers, tied to numbers already validated in w05/w06
- [x] Cost/value thinking (value-weighted queue vs. raw probability, with a concrete example)
- [x] Exports the queue and a figure to `work/outputs/` and `work/figures/`
- [x] Keeps the plan practical and explicitly non-production
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.